# Data Cleaning and Preparation

## Objective
This notebook cleans the raw DataCo Supply Chain dataset and prepares it for exploratory analysis and modeling.

## Goals
- standardize column names
- fix data types
- convert date columns
- handle missing values
- review duplicate rows
- inspect identifier-like and irrelevant columns
- flag potential leakage columns
- save a cleaned dataset for later notebooks

## Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

## Load the raw dataset

In [2]:
df = pd.read_csv("../data/raw/DataCoSupplyChainDataset.csv", encoding="latin1")
df.head()

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,Customer Country,Customer Email,Customer Fname,Customer Id,Customer Lname,Customer Password,Customer Segment,Customer State,Customer Street,Customer Zipcode,Department Id,Department Name,Latitude,Longitude,Market,Order City,Order Country,Order Customer Id,order date (DateOrders),Order Id,Order Item Cardprod Id,Order Item Discount,Order Item Discount Rate,Order Item Id,Order Item Product Price,Order Item Profit Ratio,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Cally,20755,Holloway,XXXXXXXXX,Consumer,PR,5365 Noble Nectar Island,725.0,2,Fitness,18.251453,-66.037056,Pacific Asia,Bekasi,Indonesia,20755,1/31/2018 22:56,77202,1360,13.110000,0.04,180517,327.75,0.29,1,327.75,314.640015,91.250000,Southeast Asia,Java Occidental,COMPLETE,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Irene,19492,Luna,XXXXXXXXX,Consumer,PR,2679 Rustic Loop,725.0,2,Fitness,18.279451,-66.037064,Pacific Asia,Bikaner,India,19492,1/13/2018 12:27,75939,1360,16.389999,0.05,179254,327.75,-0.80,1,327.75,311.359985,-249.089996,South Asia,Rajastán,PENDING,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,EE. UU.,XXXXXXXXX,Gillian,19491,Maldonado,XXXXXXXXX,Consumer,CA,8510 Round Bear Gate,95125.0,2,Fitness,37.292233,-121.881279,Pacific Asia,Bikaner,India,19491,1/13/2018 12:06,75938,1360,18.030001,0.06,179253,327.75,-0.80,1,327.75,309.720001,-247.779999,South Asia,Rajastán,CLOSED,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,EE. UU.,XXXXXXXXX,Tana,19490,Tate,XXXXXXXXX,Home Office,CA,3200 Amber Bend,90027.0,2,Fitness,34.125946,-118.291016,Pacific Asia,Townsville,Australia,19490,1/13/2018 11:45,75937,1360,22.940001,0.07,179252,327.75,0.08,1,327.75,304.809998,22.860001,Oceania,Queensland,COMPLETE,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Orli,19489,Hendricks,XXXXXXXXX,Corporate,PR,8671 Iron Anchor Corners,725.0,2,Fitness,18.253769,-66.037048,Pacific Asia,Townsville,Australia,19489,1/13/2018 11:24,75936,1360,29.500000,0.09,179251,327.75,0.45,1,327.75,298.250000,134.210007,Oceania,Queensland,PENDING_PAYMENT,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


In [3]:
df.shape

(180519, 53)

In [4]:
df_cleaned = df.copy()
df_cleaned.info()

<class 'pandas.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 53 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Type                           180519 non-null  str    
 1   Days for shipping (real)       180519 non-null  int64  
 2   Days for shipment (scheduled)  180519 non-null  int64  
 3   Benefit per order              180519 non-null  float64
 4   Sales per customer             180519 non-null  float64
 5   Delivery Status                180519 non-null  str    
 6   Late_delivery_risk             180519 non-null  int64  
 7   Category Id                    180519 non-null  int64  
 8   Category Name                  180519 non-null  str    
 9   Customer City                  180519 non-null  str    
 10  Customer Country               180519 non-null  str    
 11  Customer Email                 180519 non-null  str    
 12  Customer Fname                 180519 non

## Standardize cloumn names

In [5]:
def clean_column_name(col):
    col = col.strip().lower()
    col = col.replace("(", "").replace(")", "")
    col = col.replace("/", "_")
    col = col.replace("-", "_")
    col = col.replace(" ", "_")
    col = re.sub(r"[^a-z0-9_]", "", col)
    col = re.sub(r"_+", "_", col)
    return col

df_cleaned.columns = [clean_column_name(col) for col in df_cleaned.columns]
df_cleaned.columns.tolist()

['type',
 'days_for_shipping_real',
 'days_for_shipment_scheduled',
 'benefit_per_order',
 'sales_per_customer',
 'delivery_status',
 'late_delivery_risk',
 'category_id',
 'category_name',
 'customer_city',
 'customer_country',
 'customer_email',
 'customer_fname',
 'customer_id',
 'customer_lname',
 'customer_password',
 'customer_segment',
 'customer_state',
 'customer_street',
 'customer_zipcode',
 'department_id',
 'department_name',
 'latitude',
 'longitude',
 'market',
 'order_city',
 'order_country',
 'order_customer_id',
 'order_date_dateorders',
 'order_id',
 'order_item_cardprod_id',
 'order_item_discount',
 'order_item_discount_rate',
 'order_item_id',
 'order_item_product_price',
 'order_item_profit_ratio',
 'order_item_quantity',
 'sales',
 'order_item_total',
 'order_profit_per_order',
 'order_region',
 'order_state',
 'order_status',
 'order_zipcode',
 'product_card_id',
 'product_category_id',
 'product_description',
 'product_image',
 'product_name',
 'product_price',

> Column names have been standardized to lowercase snake_case format to improve readability and make coding more consistent in later notebooks.

In [6]:
df_cleaned.info()

<class 'pandas.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 53 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   type                         180519 non-null  str    
 1   days_for_shipping_real       180519 non-null  int64  
 2   days_for_shipment_scheduled  180519 non-null  int64  
 3   benefit_per_order            180519 non-null  float64
 4   sales_per_customer           180519 non-null  float64
 5   delivery_status              180519 non-null  str    
 6   late_delivery_risk           180519 non-null  int64  
 7   category_id                  180519 non-null  int64  
 8   category_name                180519 non-null  str    
 9   customer_city                180519 non-null  str    
 10  customer_country             180519 non-null  str    
 11  customer_email               180519 non-null  str    
 12  customer_fname               180519 non-null  str    
 13  customer_i

## Identify date cloumns and convert them into datetime columns

In [7]:
date_like_cols = [col for col in df_cleaned.columns if "date" in col]
date_like_cols

['order_date_dateorders', 'shipping_date_dateorders']

In [8]:
for col in date_like_cols:
    df_cleaned[col] = pd.to_datetime(df_cleaned[col], errors="coerce")
df_cleaned.info()

<class 'pandas.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 53 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   type                         180519 non-null  str           
 1   days_for_shipping_real       180519 non-null  int64         
 2   days_for_shipment_scheduled  180519 non-null  int64         
 3   benefit_per_order            180519 non-null  float64       
 4   sales_per_customer           180519 non-null  float64       
 5   delivery_status              180519 non-null  str           
 6   late_delivery_risk           180519 non-null  int64         
 7   category_id                  180519 non-null  int64         
 8   category_name                180519 non-null  str           
 9   customer_city                180519 non-null  str           
 10  customer_country             180519 non-null  str           
 11  customer_email               180519 n

In [9]:
df_cleaned[date_like_cols].isnull().sum()

order_date_dateorders       0
shipping_date_dateorders    0
dtype: int64

In [10]:
df_cleaned[date_like_cols].dtypes

order_date_dateorders       datetime64[us]
shipping_date_dateorders    datetime64[us]
dtype: object

Date-related columns were converted to datetime format using `errors="coerce"` to safely handle invalid date values.

## Handle Missing Values

In [11]:
missing_values = pd.DataFrame(
    {"Missing Count": df_cleaned.isnull().sum(),
     "Missing %": df_cleaned.isnull().mean() * 100}
).sort_values(by="Missing %", ascending=False)
missing_values[missing_values["Missing %"] > 0]

,Missing Count,Missing %
product_description,180519,100.000000
order_zipcode,155679,86.239676
customer_lname,8,0.004432
customer_zipcode,3,0.001662


In [12]:
df_cleaned.drop(columns=["product_description"], inplace=True)

`product_description` was dropped because it contains no usable information.

In [13]:
df_cleaned.drop(columns=["order_zipcode"], inplace=True)

`order_zipcode` was dropped because it can be replaced by other geographical cloumns like order region.

In [14]:
df_cleaned["customer_lname"] = df_cleaned["customer_lname"].fillna("unknown")
df_cleaned["customer_zipcode"] = df_cleaned["customer_zipcode"].fillna(df_cleaned["customer_zipcode"].mode()[0])

Impute missing values with a constant like unknown or the mode of that column

## Review duplicate values

In [15]:
df_cleaned.duplicated().sum()

np.int64(0)

There is no duplicate record present

In [16]:
df_cleaned.head()

,type,days_for_shipping_real,days_for_shipment_scheduled,benefit_per_order,sales_per_customer,delivery_status,late_delivery_risk,category_id,category_name,customer_city,customer_country,customer_email,customer_fname,customer_id,customer_lname,customer_password,customer_segment,customer_state,customer_street,customer_zipcode,department_id,department_name,latitude,longitude,market,order_city,order_country,order_customer_id,order_date_dateorders,order_id,order_item_cardprod_id,order_item_discount,order_item_discount_rate,order_item_id,order_item_product_price,order_item_profit_ratio,order_item_quantity,sales,order_item_total,order_profit_per_order,order_region,order_state,order_status,product_card_id,product_category_id,product_image,product_name,product_price,product_status,shipping_date_dateorders,shipping_mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Cally,20755,Holloway,XXXXXXXXX,Consumer,PR,5365 Noble Nectar Island,725.0,2,Fitness,18.251453,-66.037056,Pacific Asia,Bekasi,Indonesia,20755,2018-01-31 22:56:00,77202,1360,13.110000,0.04,180517,327.75,0.29,1,327.75,314.640015,91.250000,Southeast Asia,Java Occidental,COMPLETE,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-02-03 22:56:00,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Irene,19492,Luna,XXXXXXXXX,Consumer,PR,2679 Rustic Loop,725.0,2,Fitness,18.279451,-66.037064,Pacific Asia,Bikaner,India,19492,2018-01-13 12:27:00,75939,1360,16.389999,0.05,179254,327.75,-0.80,1,327.75,311.359985,-249.089996,South Asia,Rajastán,PENDING,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-18 12:27:00,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,EE. UU.,XXXXXXXXX,Gillian,19491,Maldonado,XXXXXXXXX,Consumer,CA,8510 Round Bear Gate,95125.0,2,Fitness,37.292233,-121.881279,Pacific Asia,Bikaner,India,19491,2018-01-13 12:06:00,75938,1360,18.030001,0.06,179253,327.75,-0.80,1,327.75,309.720001,-247.779999,South Asia,Rajastán,CLOSED,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-17 12:06:00,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,EE. UU.,XXXXXXXXX,Tana,19490,Tate,XXXXXXXXX,Home Office,CA,3200 Amber Bend,90027.0,2,Fitness,34.125946,-118.291016,Pacific Asia,Townsville,Australia,19490,2018-01-13 11:45:00,75937,1360,22.940001,0.07,179252,327.75,0.08,1,327.75,304.809998,22.860001,Oceania,Queensland,COMPLETE,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-16 11:45:00,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Orli,19489,Hendricks,XXXXXXXXX,Corporate,PR,8671 Iron Anchor Corners,725.0,2,Fitness,18.253769,-66.037048,Pacific Asia,Townsville,Australia,19489,2018-01-13 11:24:00,75936,1360,29.500000,0.09,179251,327.75,0.45,1,327.75,298.250000,134.210007,Oceania,Queensland,PENDING_PAYMENT,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-15 11:24:00,Standard Class


## Inspect Identifier type or Irrelevant columns

In [17]:
df_cleaned.nunique().sort_values(ascending=False).head(20)

order_item_id               180519
order_date_dateorders        65752
order_id                     65752
shipping_date_dateorders     63701
benefit_per_order            21998
order_profit_per_order       21998
order_customer_id            20652
customer_id                  20652
latitude                     11250
customer_street               7458
longitude                     4487
order_city                    3597
sales_per_customer            2927
order_item_total              2927
customer_lname                1110
order_state                   1089
order_item_discount           1017
customer_zipcode               995
customer_fname                 782
customer_city                  563
dtype: int64

In [18]:
id_columns = [col for col in df_cleaned.columns if "id" in col]
id_columns

['category_id',
 'customer_id',
 'department_id',
 'order_customer_id',
 'order_id',
 'order_item_cardprod_id',
 'order_item_id',
 'product_card_id',
 'product_category_id']

Several columns behave like identifiers. These may not be directly useful for modeling in raw form, but they are retained for now until feature selection is finalized.

In [19]:
constant_cols = [col for col in df_cleaned.columns if df_cleaned[col].nunique() <= 1]
constant_cols

['customer_email', 'customer_password', 'product_status']

In [20]:
df_cleaned.drop(columns=constant_cols, inplace=True)

Columns with no variable values are dropped because they hold no meaning in modelling and EDA

## Flag potential leakage columns

In [21]:
leakage_candidates = [
    "late_delivery_risk",
    "delivery_status",
    "days_for_shipping_real",
    "days_for_shipment_scheduled",
    "order_status"
]

df_cleaned[leakage_candidates].head()

,late_delivery_risk,delivery_status,days_for_shipping_real,days_for_shipment_scheduled,order_status
0,0,Advance shipping,3,4,COMPLETE
1,1,Late delivery,5,4,PENDING
2,0,Shipping on time,4,4,CLOSED
3,0,Advance shipping,3,4,COMPLETE
4,0,Advance shipping,2,4,PENDING_PAYMENT


Some columns appear closely tied to the delivery outcome and may introduce target leakage in predictive modeling. These columns are retained in the cleaned dataset for exploratory analysis, but they will be reviewed carefully before model training.

In [22]:
for col in ["shipping_mode", "market", "order_status", "delivery_status"]:
    print(df_cleaned[col].value_counts(dropna=False).head(10))
    print("\n")

shipping_mode
Standard Class    107752
Second Class       35216
First Class        27814
Same Day            9737
Name: count, dtype: int64


market
LATAM           51594
Europe          50252
Pacific Asia    41260
USCA            25799
Africa          11614
Name: count, dtype: int64


order_status
COMPLETE           59491
PENDING_PAYMENT    39832
PROCESSING         21902
PENDING            20227
CLOSED             19616
ON_HOLD             9804
SUSPECTED_FRAUD     4062
CANCELED            3692
PAYMENT_REVIEW      1893
Name: count, dtype: int64


delivery_status
Late delivery        98977
Advance shipping     41592
Shipping on time     32196
Shipping canceled     7754
Name: count, dtype: int64




In [23]:
print("Original shape:", df.shape)
print("Cleaned shape:", df_cleaned.shape)

Original shape: (180519, 53)
Cleaned shape: (180519, 48)


We can see that we dropped 5 columns

## Save the clean dataset

In [24]:
df_cleaned.to_csv("../data/cleaned/cleaned_supply_chain_data.csv", index=False)
df_cleaned.to_parquet("../data/cleaned/cleaned_supply_chain_data.parquet", index=False)